* Brandon Connors
* 10/10/21
* Jupyter Notebooks - Web Scraping Assignment

In [33]:
import urllib3
import re
import json
import pyodbc

# Opens the credentials file I've got saved and stores the username and password in memory
with open('C:\\School\\2021 - 4 - Fall\\DA 320\\credentials.json') as f:
    data = json.load(f)
    username = data['username']
    password = data['password']

# connects to the SQL server using the credent
conn = pyodbc.connect('Driver={SQL Server};'
                      'Server=BRANDON;'
                      'Database=DA_320;'
                      'UID=' + username + ';'
                      'PWD=' + password)


# I'm using this example code from https://zetcode.com/python/urllib3/ and tweaking it according to the feedback on the discussion boards about the security on the website.
http = urllib3.PoolManager()

# Website I'm currently scraping from.
url = 'https://www.metacritic.com/browse/movies/score/metascore/year/filtered?year_selected=2003&sort=desc&view=detailed'

# This line fakes a web browser to circumvent the security on Metacritic - This was found on stackoverflow by Sav Bell and shared with the discussion board
resp = http.request('GET', url, headers={'User-Agent': 'Mozilla/5.0'})

# This decodes the result from the request above using utf-8
text = resp.data.decode('utf-8')

# Here I am searching for the regex strings and assigning the results to lists
movieTitles = re.findall('class=\"title\"><h3>(.*?)<\/h3><\/a>', text)
releaseDates = re.findall('<div class=\"clamp-details\">\n+\s+<span>(.*?)<\/span>', text)
movieDescriptions = re.findall('<div class=\"summary\">\n\s+(.*?)\n\s+<\/div>', text)
metacriticScores = re.findall('<span class=\"title\">Metascore:<\/span>\n\s+<a class=\"metascore_anchor\" href=\"\/movie\/.+\/critic.+\n<div class.+\">(.*?)<\/div>', text)
thumbnailURLs = re.findall('<a href=\"\/movie\/.*\"><img src=\"(.*?)\" alt=\"', text)
metacriticIDs = re.findall('<input type=\"checkbox\" id=\"(.*?)\" class=\"clamp-summary-expand\"', text)
userScores = re.findall('<a class=\"metascore_anchor\" href=\"\/movie\/.+\/user.+\n<div class.+\">(.*?)<\/div>', text)
# This regex finds pairs of titles and their associated ratings, can't just be the list of ratings because not every movie has that part in its section of code.
titleRatingPairs = re.findall('<tr>(?:[\s\S]*?<h3>)(?:(?P<Title>.*?)<\/h3>)(?:[\s\S]*?<span class=\"cert_rating )(?:(?P<Rating>.*?)\">)(?:[\s\S]*?<div class=\"summary\">\s*)[\s\S]*?<\/tr>', text)

# this creates a list of ratings that is a one-to-one match with the list of movie titles, instead of the partial list due to some movies missing a rating entirely
movieRatings = []
for i in range(len(movieTitles)):
    counter = 0
    for j in range(len(titleRatingPairs)):
        if movieTitles[i] == titleRatingPairs[j][0]:
            movieRatings.append(titleRatingPairs[j][1])
            break
        elif movieTitles[i] != titleRatingPairs[j][0]:
            counter += 1
        if counter == len(titleRatingPairs):
            movieRatings.append(None)

# Sanity check here to make sure I caught everything
print('Titles:', len(movieTitles), '\tDates:', len(releaseDates), '\tDescriptions:', len(movieDescriptions), '\tScores:', len(metacriticScores), 
      '\tURLs:', len(thumbnailURLs), '\tIDs:', len(metacriticIDs), '\tUser Scores:', len(userScores), '\tTitle-Rating:', len(titleRatingPairs), '\tRatings:', len(movieRatings))

Titles: 100 	Dates: 100 	Descriptions: 100 	Scores: 100 	URLs: 100 	IDs: 100 	User Scores: 100 	Title-Rating: 78 	Ratings: 100


In [31]:
# Copied this part from the lecture notes - uses SQL commands to insert data pulled from Metacritic into SQL database
cursor = conn.cursor()
sql = "INSERT INTO Movies (MetacriticID, Title, ReleaseDate, Rating, MetacriticScore, UserScore, ThumbnailURL, Description) values (?, ?, ?, ?, ?, ?, ?, ?)"
args = "MetacriticID", "Title", "ReleaseDate", "Rating", "MetacriticScore", "UserScore", "ThumbnailURL", "Description"

# For Loop to iterate through all the items and insert them into the appropriate spot in the database
for i in range(len(metacriticIDs)):
    args = metacriticIDs[i], movieTitles[i], releaseDates[i], movieRatings[i], metacriticScores[i], userScores[i], thumbnailURLs[i], movieDescriptions[i]
    cursor.execute(sql,args)
    cursor.commit()


# Initializing the main database list - only for use in python, not SQL
#database = []
# Iterating through each item the lists above to grab each corresponding item and putting them into a list together, then putting that movie's list into the database list
#for i in range(len(movieTitles)):
#    newAddition = []
#    newAddition.append(metacriticIDs[i])
#    newAddition.append(movieTitles[i])
#    newAddition.append(releaseDates[i])
#    newAddition.append(movieRatings[i])
#    newAddition.append(metacriticScores[i])
#    newAddition.append(userScores[i])
#    newAddition.append(thumbnailURLs[i])
#    newAddition.append(movieDescriptions[i])
#    database.append(newAddition)
    
# This takes each movie's list from the database and prints each item on a new line, then skips a line before the next movie
#for i in range(len(database)):
#    for j in range(len(database[i])):
#        print(database[i][j])
#    print('Title:\t\t\t', database[i][0], sep='')
#    print('Release Date:\t\t', database[i][1], sep='')
#    print('Metacritic Score:\t', database[i][2], sep='')
#    print('Thumbnail URL:\t\t', database[i][3], sep='')
#    print('Description:\t\t', database[i][4], sep='')
#    print()